# Day 4-3: Class Imbalance 처리

**강의 시간**: 2시간  
**학습 목표**:
- Class Weights 적용
- Focal Loss 구현
- 균형 잡힌 성능 달성
- COVID Recall 95%+ 복원

**사전 요구사항**: Day 4-2 완료  
**Day 4-2 성능**: Accuracy 90.67%, COVID Recall 88.93%

## 🔧 0. 환경 설정 (Day 4-2 계속)

In [ ]:
# 라이브러리 설치
%pip install -q 'mlflow>=2,<3' dagshub tensorflow opencv-python scikit-learn

print("✅ 라이브러리 설치 완료!")

In [ ]:
# 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    classification_report, confusion_matrix,
    balanced_accuracy_score, recall_score, f1_score
)
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import mlflow
import dagshub

np.random.seed(42)
tf.random.set_seed(42)

print(f"✅ TensorFlow {tf.__version__}")

In [ ]:
# 시각화 설정
sns.set_style('whitegrid')

# 1) 폰트 파일 직접 다운로드 (런타임 재시작 불필요)
!wget -q -O NanumGothic.ttf -L "https://fonts.gstatic.com/ea/nanumgothic/v5/NanumGothic-Regular.ttf"

import matplotlib.font_manager as fm

# 폰트 파일 경로
font_path = "NanumGothic.ttf"

# 폰트 매니저에 폰트 추가
fm.fontManager.addfont(font_path)

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

# 폰트 속성 설정
font_prop = fm.FontProperties(fname=font_path)
plt.rcParams["font.family"] = font_prop.get_name()
plt.rcParams["axes.unicode_minus"] = False

🔥 이 부분은 수정이 필요합니다.

**repo_owner**와 **repo_name**을 본인의 Dagshub 정보로 채워 주세요.

In [ ]:
# MLflow 설정
import mlflow
import dagshub

repo_owner = # 🔥 직접 작성이 필요합니다.
repo_name  = # 🔥 직접 작성이 필요합니다.

dagshub.init(repo_owner=repo_owner, repo_name=repo_name, mlflow=True)
mlflow.set_experiment('day4-covid-xray-classification')
print('✅ MLflow 설정 완료!')

## 📂 1. 데이터 로드 (Day 4-2 계속)

## 📦 1. 데이터 다운로드

### Kaggle API 설정

**사전 준비**:
1. Kaggle 계정 생성 (https://www.kaggle.com)
2. Account → API → "Create New API Token"
3. `kaggle.json` 다운로드

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_API_TOKEN')
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

try:
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()

    dataset_id = 'tawsifurrahman/covid19-radiography-database'
    print(f"📥 {dataset_id} 다운로드 시작...")

    api.dataset_download_files(
        dataset_id,
        path='./data',
        unzip=True,
        quiet=False
    )

    print("
✅ 다운로드 및 압축 해제 완료!")

except Exception as e:
    print(f"
❌ 오류 발생: {e}")

In [ ]:
# 데이터 구조 확인
data_dir = Path('./data/COVID-19_Radiography_Dataset')

In [ ]:
def get_file_paths_and_labels(data_dir):
    """이미지 경로와 라벨만 수집 (메모리 효율적)"""

    class_names = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}

    file_paths = []
    labels = []

    print("📋 파일 경로 수집 중...")

    for class_name in class_names:
        image_dir = data_dir / class_name / 'images'
        image_paths = list(image_dir.glob('*.png'))

        for img_path in image_paths:
            file_paths.append(str(img_path))
            labels.append(class_to_idx[class_name])

        print(f"  {class_name:20s}: {len(image_paths):5d}개")

    print(f"
✅ 총 {len(file_paths):,}개 파일 경로 수집 완료!")

    return file_paths, labels, class_names

file_paths, labels, class_names = get_file_paths_and_labels(data_dir)

print(f"
데이터 정보:")
print(f"  파일 수: {len(file_paths):,}개")
print(f"  클래스: {class_names}")

In [ ]:
from sklearn.model_selection import train_test_split

train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

print(f"Train: {len(train_paths):,}개")
print(f"Val  : {len(val_paths):,}개")

## ⚖️ 2. Class Weights 계산

In [ ]:
train_dist = pd.Series(train_labels).value_counts().sort_index()

print("="*60)
print("  Train Class Distribution")
print("="*60)
for idx, count in train_dist.items():
    print(f"{class_names[idx]:20s}: {count:5d} ({count/len(train_labels)*100:5.1f}%)")
print("="*60)

plt.figure(figsize=(10, 6))
bars = plt.bar(class_names, train_dist.values,
               color=['red', 'orange', 'green', 'purple'], edgecolor='black')
plt.ylabel('이미지 수', fontweight='bold', fontsize=12)
plt.title('Train Set 클래스 분포 (Imbalanced)', fontweight='bold', fontsize=14)
plt.xticks(rotation=15)

for bar, count in zip(bars, train_dist.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{count}', ha='center', fontweight='bold')

plt.axhline(len(train_labels)/4, color='red', linestyle='--',
            linewidth=2, label='균형 분포 (25%)')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('class_distribution_train.png', dpi=100, bbox_inches='tight')
plt.show()

🔥 이 부분을 같이 작성해봅시다.

**`compute_class_weight`** 를 사용해 클래스 가중치를 계산하세요.  
`class_weight='balanced'`, `classes=np.unique(train_labels)`, `y=train_labels`

In [ ]:
# Class Weights 계산
class_weights_array = # 🔥 직접 작성이 필요합니다.

class_weights = {i: weight for i, weight in enumerate(class_weights_array)}

print("
계산된 Class Weights:")
print("="*60)
for idx, weight in class_weights.items():
    print(f"{class_names[idx]:20s}: {weight:.4f}")
print("="*60)

plt.figure(figsize=(10, 6))
bars = plt.bar(class_names, class_weights.values(),
               color=['red', 'orange', 'green', 'purple'], edgecolor='black')
plt.ylabel('Weight', fontweight='bold', fontsize=12)
plt.title('Class Weights (Inverse Frequency)', fontweight='bold', fontsize=14)
plt.xticks(rotation=15)

for bar, weight in zip(bars, class_weights.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{weight:.2f}', ha='center', fontweight='bold')

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('class_weights.png', dpi=100, bbox_inches='tight')
plt.show()

print("
💡 해석:")
print(f"   Viral (가장 희소): Weight={class_weights[3]:.2f} (가장 높음)")
print(f"   Normal (가장 많음): Weight={class_weights[2]:.2f} (가장 낮음)")

## 📊 3. Dataset 생성

In [ ]:
def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.resize(img, (224, 224))
    img = tf.image.grayscale_to_rgb(img)
    img = preprocess_input(img)
    return img, label

BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_dataset = train_dataset.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
train_dataset = train_dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_dataset = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_dataset = val_dataset.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("✅ Dataset 생성 완료!")

## 🏗️ 4. 모델 구축 함수

In [ ]:
def build_resnet50_model():
    """ResNet50 Transfer Learning 모델"""

    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = False

    inputs = keras.Input(shape=(224, 224, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(4, activation='softmax')(x)

    model = Model(inputs, outputs, name='ResNet50_Transfer')
    return model

print("✅ 모델 구축 함수 정의 완료!")

## 🧪 5. Experiment 1: Class Weights 적용

In [ ]:
model_weighted = build_resnet50_model()

model_weighted.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ 모델 컴파일 완료!")

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7
)

print("✅ Callbacks 설정 완료!")

🔥 이 부분은 수정이 필요합니다.

**run_name**을 채우고, `model.fit()`에 **class_weight** 파라미터를 추가하세요.  
`class_weight=class_weights` 한 줄로 Class Weights가 적용됩니다.

In [ ]:
try:
    mlflow.end_run()
except:
    pass

with mlflow.start_run(run_name=''):  # 🔥 직접 작성이 필요합니다. (예: ResNet50_ClassWeights)
    mlflow.log_params({
        'model': 'ResNet50',
        'strategy': 'Class Weights',
        'weights': str(class_weights),
        'epochs': 15,
        'batch_size': BATCH_SIZE
    })

    print("🏃 Class Weights 적용 학습 시작...
")

    history_weighted = model_weighted.fit(
        train_dataset,
        epochs=15,
        validation_data=val_dataset,
        class_weight=# 🔥 직접 작성이 필요합니다.,  # ← Class Weights!
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )

    final_loss, final_acc = model_weighted.evaluate(val_dataset, verbose=0)

    mlflow.log_metrics({
        'val_loss': final_loss,
        'val_accuracy': final_acc
    })

    model_weighted.save('resnet50_class_weights.h5')
    mlflow.keras.log_model(model_weighted, 'model')

    print(f"
{'='*60}")
    print("  Class Weights 학습 완료")
    print('='*60)
    print(f"  Val Accuracy: {final_acc:.4f} ({final_acc*100:.2f}%)")
    print('='*60)

## 🔥 6. Experiment 2: Focal Loss

🔥 이 부분을 같이 작성해봅시다.

Focal Loss의 핵심인 **Modulating Factor**와 **최종 loss 수식**을 완성해 보세요.

```
FL(p_t) = -α × (1 - p_t)^γ × log(p_t)

focal_term = (1 - p_t)^γ  ← Easy examples의 loss를 줄이는 핵심
loss       = α × focal_term × CE
```

In [ ]:
def focal_loss(gamma=2., alpha=0.25):
    """
    Focal Loss for multi-class classification
    FL(p_t) = -α * (1 - p_t)^γ * log(p_t)
    """
    def focal_loss_fixed(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_true_one_hot = tf.one_hot(y_true, depth=y_pred.shape[-1])

        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)

        # Cross entropy
        ce = -y_true_one_hot * tf.math.log(y_pred)

        # Focal term: (1 - p_t)^gamma
        p_t = tf.reduce_sum(y_true_one_hot * y_pred, axis=-1, keepdims=True)
        focal_term = # 🔥 직접 작성이 필요합니다. (tf.pow(1. - p_t, gamma))

        # Final loss
        loss = # 🔥 직접 작성이 필요합니다. (alpha * focal_term * tf.reduce_sum(ce, axis=-1))

        return tf.reduce_mean(loss)

    return focal_loss_fixed

print("✅ Focal Loss 구현 완료!")

In [ ]:
model_focal = build_resnet50_model()

model_focal.compile(
    optimizer='adam',
    loss=focal_loss(gamma=2.0, alpha=0.25),
    metrics=['accuracy']
)

print("✅ Focal Loss 모델 컴파일 완료!")

🔥 이 부분은 수정이 필요합니다.

**Focal Loss 실험**의 **run_name**을 채워주세요.

In [ ]:
try:
    mlflow.end_run()
except:
    pass

with mlflow.start_run(run_name=''):  # 🔥 직접 작성이 필요합니다. (예: ResNet50_FocalLoss)
    mlflow.log_params({
        'model': 'ResNet50',
        'strategy': 'Focal Loss',
        'gamma': 2.0,
        'alpha': 0.25,
        'epochs': 15,
        'batch_size': BATCH_SIZE
    })

    print("🏃 Focal Loss 학습 시작...
")

    history_focal = model_focal.fit(
        train_dataset,
        epochs=15,
        validation_data=val_dataset,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )

    final_loss, final_acc = model_focal.evaluate(val_dataset, verbose=0)

    mlflow.log_metrics({
        'val_loss': final_loss,
        'val_accuracy': final_acc
    })

    model_focal.save('resnet50_focal_loss.h5')
    mlflow.keras.log_model(model_focal, 'model')

    print(f"
{'='*60}")
    print("  Focal Loss 학습 완료")
    print('='*60)
    print(f"  Val Accuracy: {final_acc:.4f} ({final_acc*100:.2f}%)")
    print('='*60)

## 📊 7. 성능 비교 & 분석

In [ ]:
def predict_and_evaluate(model, dataset, model_name):
    """모델 예측 및 평가"""
    print(f"
🔮 {model_name} 예측 중...")

    y_true, y_pred = [], []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_true.extend(labels.numpy())
        y_pred.extend(np.argmax(preds, axis=1))

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    acc = np.mean(y_true == y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred, average=None)
    f1 = f1_score(y_true, y_pred, average='macro')

    print(f"  Accuracy: {acc:.4f}")
    print(f"  Balanced Accuracy: {balanced_acc:.4f}")
    print(f"  Macro F1: {f1:.4f}")

    return y_true, y_pred, {
        'accuracy': acc,
        'balanced_accuracy': balanced_acc,
        'covid_recall': recall[0],
        'macro_f1': f1
    }

# Baseline (Day 4-2 결과 사용)
baseline_metrics = {
    'accuracy': 0.9067,
    'balanced_accuracy': 0.9035,
    'covid_recall': 0.8893,
    'macro_f1': 0.9148
}

_, _, weighted_metrics = predict_and_evaluate(
    model_weighted, val_dataset, "Class Weights"
)

_, _, focal_metrics = predict_and_evaluate(
    model_focal, val_dataset, "Focal Loss"
)

print("
✅ 모든 예측 완료!")

In [ ]:
comparison_df = pd.DataFrame({
    'Model': ['Baseline', 'Class Weights', 'Focal Loss'],
    'Accuracy': [
        baseline_metrics['accuracy'],
        weighted_metrics['accuracy'],
        focal_metrics['accuracy']
    ],
    'Balanced Acc': [
        baseline_metrics['balanced_accuracy'],
        weighted_metrics['balanced_accuracy'],
        focal_metrics['balanced_accuracy']
    ],
    'COVID Recall': [
        baseline_metrics['covid_recall'],
        weighted_metrics['covid_recall'],
        focal_metrics['covid_recall']
    ],
    'Macro F1': [
        baseline_metrics['macro_f1'],
        weighted_metrics['macro_f1'],
        focal_metrics['macro_f1']
    ]
})

print("
" + "="*80)
print("  성능 비교")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['Accuracy', 'Balanced Acc', 'COVID Recall', 'Macro F1']
data = [
    [baseline_metrics['accuracy'], weighted_metrics['accuracy'], focal_metrics['accuracy']],
    [baseline_metrics['balanced_accuracy'], weighted_metrics['balanced_accuracy'], focal_metrics['balanced_accuracy']],
    [baseline_metrics['covid_recall'], weighted_metrics['covid_recall'], focal_metrics['covid_recall']],
    [baseline_metrics['macro_f1'], weighted_metrics['macro_f1'], focal_metrics['macro_f1']]
]

models = ['Baseline', 'Class Weights', 'Focal Loss']
colors = ['steelblue', 'coral', 'mediumseagreen']

for i, (ax, metric, values) in enumerate(zip(axes.flat, metrics, data)):
    bars = ax.bar(models, values, color=colors, edgecolor='black')
    ax.set_ylabel(metric, fontweight='bold')
    ax.set_title(metric, fontweight='bold', fontsize=12)
    ax.set_ylim(0.85, 1.0)
    ax.grid(axis='y', alpha=0.3)

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', fontweight='bold', fontsize=9)

plt.suptitle('Class Imbalance 처리 방법 비교',
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('imbalance_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## 🏆 8. 최종 모델 선정 & 평가

In [ ]:
covid_recalls = {
    'Baseline': baseline_metrics['covid_recall'],
    'Class Weights': weighted_metrics['covid_recall'],
    'Focal Loss': focal_metrics['covid_recall']
}

best_strategy = max(covid_recalls, key=covid_recalls.get)
print(f"🏆 Best Strategy: {best_strategy}")
print(f"   COVID Recall: {covid_recalls[best_strategy]:.4f}")

if best_strategy == 'Class Weights':
    best_model = model_weighted
elif best_strategy == 'Focal Loss':
    best_model = model_focal

print(f"
✅ {best_strategy} 모델 선정!")

In [ ]:
y_true, y_pred, _ = predict_and_evaluate(best_model, val_dataset, f"Best ({best_strategy})")

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.xlabel('예측', fontweight='bold', fontsize=12)
plt.ylabel('실제', fontweight='bold', fontsize=12)
plt.title(f'Confusion Matrix — {best_strategy}',
          fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(f'confusion_matrix_{best_strategy.lower().replace(" ", "_")}.png',
            dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
report = classification_report(y_true, y_pred,
                              target_names=class_names,
                              digits=4)

print("
" + "="*60)
print(f"  Classification Report — {best_strategy}")
print("="*60)
print(report)
print("="*60)

from sklearn.metrics import precision_recall_fscore_support
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None
)

print("
클래스별 성능:")
print("-"*60)
for i, class_name in enumerate(class_names):
    print(f"{class_name:20s}: "
          f"Precision={precision[i]:.4f}, "
          f"Recall={recall[i]:.4f}, "
          f"F1={f1[i]:.4f}")
print("-"*60)

covid_idx = 0
print(f"
⚠️ COVID-19 Recall: {recall[covid_idx]:.4f} ({recall[covid_idx]*100:.2f}%)")
print(f"   → {int(recall[covid_idx]*support[covid_idx])}/{support[covid_idx]} COVID 환자 탐지")

if recall[covid_idx] >= 0.95:
    print("
🎉 목표 달성! COVID Recall 95%+ 성공!")
else:
    print(f"
⚠️ 목표 미달: {0.95 - recall[covid_idx]:.2%}p 부족")

## 🧠 9. 핵심 개념 정리

### 오늘 배운 것

**1. Class Imbalance 문제**
- Normal 48% vs Viral 6% (7.6:1)
- 모델이 다수 클래스에 bias
- Overall Accuracy는 높지만 소수 클래스 성능 낮음

**2. Class Weights**
- Inverse Frequency: 희소 클래스에 높은 가중치
- Loss 계산 시 적용: `loss = weight * CE`
- 구현: `model.fit(..., class_weight=weights)`

**3. Focal Loss**
- Hard examples에 집중
- Easy examples의 loss 감소
- `FL = -α * (1-p_t)^γ * log(p_t)`

**4. 성능 향상**
```
COVID Recall:
  Baseline (Day 4-2): 88.93%
  + Class Weights:    95%+
  
→ False Negative 50% 감소!
→ 생명 구함!
```

**5. Trade-offs**
- Overall Accuracy 약간 감소 가능
- BUT 모든 클래스 균형 잡힌 성능
- 의료 AI: Recall > Accuracy

---

### Day 4-4 예고

**Grad-CAM & 최종 평가**
- Grad-CAM으로 모델 해석
- 어디를 보는지 시각화
- 의료 AI 설명 가능성
- 최종 종합 평가

축하합니다! Day 4-3 완료! 🎉

## ✅ Day 4-3 완료 체크리스트

- [ ] Class Imbalance 분석
- [ ] Class Weights 계산
- [ ] Class Weights 학습
- [ ] Focal Loss 구현
- [ ] Focal Loss 학습
- [ ] 3가지 전략 비교
- [ ] Balanced Accuracy 확인
- [ ] COVID Recall 95%+ 달성
- [ ] Per-class 성능 균형 확인
- [ ] Best 모델 선정
- [ ] Confusion Matrix 분석
- [ ] MLflow 기록 완료

## 🎯 다음 단계 (Day 4-4)

**Day 4-4: Grad-CAM & 최종 평가**

**내용:**
- Grad-CAM 구현
- Activation Map 시각화
- 모델이 어디를 보는지 확인
- 의료 AI 설명 가능성
- 최종 종합 평가 리포트

**목표:**
- 모델 해석 가능성 확보
- 의사가 이해할 수 있는 설명
- Day 4 완성!

수고하셨습니다! 🚀